In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. File configuration
# ============================================================

# # Example 1: Training dataset
# input_path = "data/data_all.xlsx"
# input_sheet = "MCs"
# output_path = "data/training_features.csv"

# Example 2: Candidate screening space
input_path = "five_metal_compounds.csv"
input_sheet = None
output_path = "data/five_metal_compounds_features.csv"

descriptor_path = "data/data_all.xlsx"
metal_descriptor_sheet = "M_descriptors"
nonmetal_descriptor_sheet = "C_descriptors"

formula_column = "compound"
current_density_column = "CurrentDensity"
target_column = "overpotential"

In [2]:
# ============================================================
# 2. Load the sample dataset and elemental descriptors
# ============================================================
input_suffix = Path(input_path).suffix.lower()

if input_suffix == ".csv":
    df = pd.read_csv(input_path)

elif input_suffix in [".xlsx", ".xls"]:
    df = pd.read_excel(
        input_path,
        sheet_name=input_sheet
    )

else:
    raise ValueError(
        f"Unsupported input file format: {input_suffix}"
    )

M_desc_raw = pd.read_excel(
    descriptor_path,
    sheet_name=metal_descriptor_sheet
)

C_desc_raw = pd.read_excel(
    descriptor_path,
    sheet_name=nonmetal_descriptor_sheet
)

In [3]:
# ============================================================
# 3. Normalize an elemental descriptor table
# ============================================================
def normalize_desc_table(
    desc: pd.DataFrame,
    element_col="element"
):
    """
    Normalize an elemental descriptor table and use element
    symbols as the row index.
    """
    desc = desc.copy()

    # Extract valid element symbols
    desc[element_col] = (
        desc[element_col]
        .astype(str)
        .str.strip()
        .str.extract(
            r"([A-Z][a-z]?)",
            expand=False
        )
    )

    desc = (
        desc
        .dropna(subset=[element_col])
        .set_index(element_col)
    )

    # Replace missing-value symbols with NaN
    object_columns = desc.select_dtypes(
        include=["object"]
    ).columns

    if len(object_columns) > 0:
        desc[object_columns] = desc[
            object_columns
        ].replace(
            {
                "/": np.nan,
                "\uff0f": np.nan
            }
        )

    # Convert descriptor values to numeric values
    for column in desc.columns:
        desc[column] = pd.to_numeric(
            desc[column],
            errors="coerce"
        )

    # Retain the first record for duplicated elements
    desc = desc[
        ~desc.index.duplicated(keep="first")
    ]

    return desc

In [4]:
# ============================================================
# 4. Parse a chemical formula
# ============================================================
def parse_formula(formula: str):
    """
    Parse a chemical formula into element symbols and
    stoichiometric coefficients.
    """
    formula = str(formula).strip()

    pattern = r"([A-Z][a-z]?)([0-9]*\.?[0-9]*)"
    tokens = re.findall(pattern, formula)

    if not tokens:
        raise ValueError(
            f"Cannot parse formula: {formula}"
        )

    element_amounts = {}

    for element, number in tokens:
        amount = (
            float(number)
            if number not in ["", None]
            else 1.0
        )

        element_amounts[element] = (
            element_amounts.get(element, 0.0)
            + amount
        )

    return element_amounts


In [5]:
# ============================================================
# 5. Define feature calculation functions
# ============================================================
def mixing_entropy(fractions):
    """
    Calculate the configurational mixing entropy without
    including the gas constant.
    """
    fractions = np.asarray(
        fractions,
        dtype=float
    )

    fractions = fractions[
        fractions > 0
    ]

    if len(fractions) == 0:
        return 0.0

    return float(
        -np.sum(
            fractions * np.log(fractions)
        )
    )


def weighted_mean(fractions, values):
    """
    Calculate the composition-weighted mean of an elemental
    property while excluding missing values.
    """
    fractions = np.asarray(
        fractions,
        dtype=float
    )

    values = np.asarray(
        values,
        dtype=float
    )

    valid_mask = ~np.isnan(values)

    if valid_mask.sum() == 0:
        return 0.0

    valid_fractions = fractions[valid_mask]
    valid_fractions = (
        valid_fractions
        / valid_fractions.sum()
    )

    valid_values = values[valid_mask]

    return float(
        np.sum(
            valid_fractions * valid_values
        )
    )


def pairwise_mismatch(fractions, values):
    """
    Calculate the composition-weighted pairwise mismatch
    of an elemental property.
    """
    fractions = np.asarray(
        fractions,
        dtype=float
    )

    values = np.asarray(
        values,
        dtype=float
    )

    valid_mask = ~np.isnan(values)

    # Pairwise mismatch requires at least two valid values
    if valid_mask.sum() < 2:
        return 0.0

    valid_fractions = (
        fractions[valid_mask]
        / fractions[valid_mask].sum()
    )

    valid_values = values[valid_mask]
    n_values = len(valid_values)

    mismatch = 0.0

    for i in range(n_values):
        for j in range(i + 1, n_values):
            mismatch += (
                valid_fractions[i]
                * valid_fractions[j]
                * abs(
                    valid_values[i]
                    - valid_values[j]
                )
            )

    return float(mismatch)


In [6]:
# ============================================================
# 6. Generate features for one chemical composition
# ============================================================
def featurize_one_general(
    formula: str,
    M_desc: pd.DataFrame,
    C_desc: pd.DataFrame,
    metal_cols,
    nonmetal_cols
):
    """
    Generate composition-based features for one compound.
    """
    features = {}
    element_amounts = parse_formula(formula)

    metals = [
        element
        for element in element_amounts
        if element in M_desc.index
    ]

    nonmetals = [
        element
        for element in element_amounts
        if element in C_desc.index
    ]

    if len(metals) == 0:
        raise ValueError(
            "No metals found in M_descriptors "
            f"for formula: {formula}"
        )

    # Calculate metal amounts and atomic fractions
    metal_amount_dict = {
        metal: float(element_amounts[metal])
        for metal in metals
    }

    metal_amounts = np.array(
        [
            metal_amount_dict[metal]
            for metal in metals
        ],
        dtype=float
    )

    metal_fractions = (
        metal_amounts
        / metal_amounts.sum()
    )

    # Record the largest metal atomic fraction
    major_metal_index = int(
        np.argmax(metal_amounts)
    )

    features["metal_x_max"] = float(
        metal_fractions[major_metal_index]
    )

    # Rank metals by decreasing stoichiometric coefficient.
    # Ties are resolved by increasing atomic number and
    # then alphabetical order of the element symbol.
    ranked_metals = sorted(
        metals,
        key=lambda element: (
            -metal_amount_dict[element],
            (
                M_desc.loc[element, "AN"]
                if (
                    "AN" in M_desc.columns
                    and pd.notna(
                        M_desc.loc[element, "AN"]
                    )
                )
                else 999
            ),
            element
        )
    )

    # Store atomic numbers in five fixed positions
    ranked_atomic_numbers = []

    for metal in ranked_metals[:5]:
        if (
            "AN" in M_desc.columns
            and pd.notna(
                M_desc.loc[metal, "AN"]
            )
        ):
            ranked_atomic_numbers.append(
                float(
                    M_desc.loc[metal, "AN"]
                )
            )
        else:
            ranked_atomic_numbers.append(0.0)

    # Pad unused positions with zeros
    while len(ranked_atomic_numbers) < 5:
        ranked_atomic_numbers.append(0.0)

    features["metal1_AN"] = ranked_atomic_numbers[0]
    features["metal2_AN"] = ranked_atomic_numbers[1]
    features["metal3_AN"] = ranked_atomic_numbers[2]
    features["metal4_AN"] = ranked_atomic_numbers[3]
    features["metal5_AN"] = ranked_atomic_numbers[4]

    # Calculate nonmetal amounts and atomic fractions
    nonmetal_amounts = (
        np.array(
            [
                element_amounts[element]
                for element in nonmetals
            ],
            dtype=float
        )
        if nonmetals
        else np.array([], dtype=float)
    )

    nonmetal_fractions = (
        nonmetal_amounts
        / nonmetal_amounts.sum()
        if len(nonmetal_amounts) > 0
        else np.array([], dtype=float)
    )

    # Calculate weighted means and mismatch descriptors
    # for metal properties
    for column in metal_cols:
        values = np.array(
            [
                M_desc.loc[metal, column]
                for metal in metals
            ],
            dtype=float
        )

        features[f"M_{column}"] = weighted_mean(
            metal_fractions,
            values
        )

        features[
            f"M_{column}_mismatch"
        ] = pairwise_mismatch(
            metal_fractions,
            values
        )

    # Calculate weighted means for nonmetal properties
    for column in nonmetal_cols:
        if len(nonmetals) == 0:
            features[f"C_{column}"] = 0.0

        else:
            values = np.array(
                [
                    C_desc.loc[element, column]
                    for element in nonmetals
                ],
                dtype=float
            )

            features[f"C_{column}"] = weighted_mean(
                nonmetal_fractions,
                values
            )

    # Calculate stoichiometric features
    total_metal_amount = float(
        metal_amounts.sum()
    )

    nitrogen_amount = float(
        element_amounts.get("N", 0.0)
    )

    oxygen_amount = float(
        element_amounts.get("O", 0.0)
    )

    sulfur_amount = float(
        element_amounts.get("S", 0.0)
    )

    total_anion_amount = (
        nitrogen_amount
        + oxygen_amount
        + sulfur_amount
    )

    features["N_metal"] = int(
        len(metals)
    )

    features["s_mix_metal"] = mixing_entropy(
        metal_fractions
    )

    features["anion_to_metal_ratio"] = (
        float(
            total_anion_amount
            / total_metal_amount
        )
        if total_metal_amount > 0
        else 0.0
    )

    return features


In [7]:
# ============================================================
# 7. Build the complete feature table
# ============================================================
def build_feature_table_general(
    df: pd.DataFrame,
    M_desc: pd.DataFrame,
    C_desc: pd.DataFrame,
    formula_col="compounds",
    j_col="CurrentDensity",
    target_col="overpotential"
):
    """
    Build a feature table for all compounds in a dataset.
    """
    metal_cols = list(M_desc.columns)
    nonmetal_cols = list(C_desc.columns)

    feature_rows = [
        featurize_one_general(
            formula,
            M_desc,
            C_desc,
            metal_cols,
            nonmetal_cols
        )
        for formula in df[formula_col].astype(str)
    ]

    feature_table = pd.DataFrame(
        feature_rows
    )

    # Insert the original compound formulas
    feature_table.insert(
        0,
        "compound",
        df[formula_col].astype(str).values
    )

    # Add current density when available
    if j_col in df.columns:
        feature_table["CurrentDensity"] = (
            pd.to_numeric(
                df[j_col],
                errors="coerce"
            )
        )

    # Store the formula as the composition identifier
    feature_table["composition_key"] = (
        df[formula_col].astype(str).values
    )

    # Add the prediction target when available
    if target_col in df.columns:
        feature_table[target_col] = pd.to_numeric(
            df[target_col],
            errors="coerce"
        )

    # Define the final feature order
    ordered_columns = [
        "compound",
        "metal_x_max",
        "metal1_AN",
        "metal2_AN",
        "metal3_AN",
        "metal4_AN",
        "metal5_AN",
        "M_AN",
        "M_AN_mismatch",
        "M_AR",
        "M_AR_mismatch",
        "M_FIE",
        "M_FIE_mismatch",
        "M_EA",
        "M_EA_mismatch",
        "M_EN",
        "M_EN_mismatch",
        "M_Nd",
        "M_Nd_mismatch",
        "M_Nv",
        "M_Nv_mismatch",
        "C_EN",
        "N_metal",
        "s_mix_metal",
        "anion_to_metal_ratio",
        "CurrentDensity",
        "composition_key",
        "overpotential"
    ]

    # Retain only columns present in the current dataset
    feature_table = feature_table[
        [
            column
            for column in ordered_columns
            if column in feature_table.columns
        ]
    ]

    return feature_table

# ============================================================
# 8. Generate and save the feature table
# ============================================================
M_desc = normalize_desc_table(
    M_desc_raw,
    element_col="element"
)

C_desc = normalize_desc_table(
    C_desc_raw,
    element_col="element"
)

X = build_feature_table_general(
    df,
    M_desc,
    C_desc,
    formula_col=formula_column,
    j_col=current_density_column,
    target_col=target_column
)

X.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Feature table saved to {output_path}. "
    f"Shape: {X.shape}"
)

print(X.head())

Feature table saved to data/five_metal_compounds_features.csv. Shape: (415800, 26)
                       compound  metal_x_max  metal1_AN  metal2_AN  metal3_AN  \
0    Mn0.2Fe0.1Co0.1Ni0.1Ti0.5N          0.5       22.0       25.0       26.0   
1  Mn0.2Fe0.15Co0.1Ni0.05Ti0.5N          0.5       22.0       25.0       26.0   
2  Mn0.2Fe0.2Co0.05Ni0.05Ti0.5N          0.5       22.0       25.0       26.0   
3    Mn0.2Fe0.1Co0.1Cu0.1Ti0.5N          0.5       22.0       25.0       26.0   
4  Mn0.2Fe0.15Co0.1Cu0.05Ti0.5N          0.5       22.0       25.0       26.0   

   metal4_AN  metal5_AN   M_AN  M_AN_mismatch    M_AR  ...  M_EN_mismatch  \
0       27.0       28.0  24.10         1.2100  1.6590  ...       0.072000   
1       27.0       28.0  24.00         1.1350  1.6625  ...       0.069100   
2       27.0       28.0  23.95         1.0975  1.6645  ...       0.067225   
3       27.0       29.0  24.20         1.3000  1.6550  ...       0.071100   
4       27.0       29.0  24.05         1.1825